# Etapa Prática 1 — Pré-processamento
Trabalho de PLN — Letras e significados de músicas (letras.mus.br)

Este notebook aplica o pipeline de limpeza e preparação sobre o campo `letra` do dataset coletado:

1. Normalização (minúsculas, remoção de ruído/pontuação)
2. Tokenização
3. Remoção de stopwords
4. Stemming (RSLP)
5. Lematização (spaCy, modelo `pt_core_news_sm`)


## 0. Base de dados: justificativa e dicionário de dados

**Justificativa da escolha:** o dataset reúne letras e significados de músicas do [letras.mus.br](https://www.letras.mus.br), cobrindo os 10 gêneros musicais mais acessados do site (Sertanejo, Pop, Rock, MPB, Funk, Rap, Pagode, Forró, Gospel, Reggae), com ~47 músicas por gênero (467 músicas no total). A base é adequada às tarefas de PLN propostas no projeto (classificação de texto, extração de informações, recuperação de informações e recomendação por similaridade de significado) por três motivos:
- **Volume**: ~467 documentos textuais, suficiente para tarefas de classificação/recuperação em escala de trabalho de disciplina.
- **Abrangência**: 10 gêneros diferentes garantem variedade de vocabulário, tema e estilo (ex: linguagem coloquial do Funk vs. poética da MPB), o que é relevante para diferenciar classes.
- **Estrutura rica**: além da letra, cada música tem um campo de *significado* (interpretação textual da letra, também coletado do site), o que abre espaço para tarefas de PLN além de classificação simples (ex: comparar letra x significado).

**Dicionário de dados** (`dataset_musicas_preprocessado.json`, lista de objetos):

| Campo | Tipo | Descrição |
|---|---|---|
| `titulo` | string | Título da música |
| `artista` | string | Nome do artista/banda |
| `metadados.generos` | lista de strings | Gêneros associados à música conforme breadcrumb da página (pode ter mais de um) |
| `metadados.genero_ranking` | string | Gênero usado para a coleta (um dos 10 gêneros-alvo) |
| `metadados.exibicoes` | string | Número de visualizações da página no site (quando disponível) |
| `metadados.url_letra` | string | URL de origem da letra |
| `metadados.url_significado` | string | URL de origem do significado |
| `letra` | string | Letra completa da música, como extraída do site (texto bruto) |
| `significados` | lista de strings | Parágrafos da interpretação/significado da letra, quando disponível no site |
| `preprocessamento.texto_normalizado` | string | Letra em minúsculas, sem pontuação/números/símbolos, espaços colapsados |
| `preprocessamento.tokens` | lista de strings | Tokens do texto normalizado, sem stopwords e sem tokens de 1 caractere |
| `preprocessamento.tokens_stemizados` | lista de strings | `tokens` após stemming (RSLP) |
| `preprocessamento.tokens_lematizados` | lista de strings | Tokens lematizados via spaCy (`pt_core_news_sm`), a partir do texto normalizado |

**Dificuldades observadas:** URLs de listagem do site (`/estilos/` vs `/mais-acessadas/`) e slugs de gênero inconsistentes (ex.: Rap está em `hip-hop-rap`, Gospel em `gospelreligioso`) exigiram ajuste manual do scraper. Também há ruído residual do site (páginas institucionais como `/academy/`) que precisou ser filtrado.


In [2]:
!python -m spacy download pt_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 93.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
# Dependências (rodar uma vez)
# !pip install nltk spacy
# !python -m spacy download pt_core_news_sm

import json
import re
import nltk
import spacy

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')  # PT e EN
nltk.download('rslp')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

nlp_pt = spacy.load('pt_core_news_sm', disable=['ner', 'parser'])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package rslp to /root/nltk_data...
[nltk_data]   Package rslp is already up-to-date!


## 1. Carregar dataset

In [2]:
with open('dataset_top10_generos_50_musicas.json', encoding='utf-8') as f:
    dataset = json.load(f)

# Descarta registros sem título/letra (lixo de scraping, ex: páginas do /academy/)
musicas = [m for m in dataset if m.get('titulo') and m.get('letra')]

print(f"Registros no arquivo: {len(dataset)}")
print(f"Músicas válidas: {len(musicas)}")

Registros no arquivo: 470
Músicas válidas: 467


## 2. Normalização e remoção de ruído
Minúsculas, remoção de pontuação/números/caracteres especiais, colapso de espaços. Mantém acentos (relevantes em PT-BR).

In [3]:
# Muitas letras do dataset têm trechos ou são inteiramente em inglês,
# então stopwords de EN também precisam ser filtradas.
# Também adicionamos coloquialismos/interjeições comuns em letra de música
# que não estão na lista padrão do NLTK (contrações do PT-BR e vocalizações).
STOPWORDS_COLOQUIAIS = {
    'pra', 'pro', 'tá', 'tô', 'cê', 'né', 'vc', 'ai', 'aí', 'eh', 'ah',
    'oh', 'uh', 'yeah', 'la', 'ei', 'uou', 'ooh', 'nana', 'lalala'
}

STOPWORDS_PT = (
    set(stopwords.words('portuguese'))
    | set(stopwords.words('english'))
    | STOPWORDS_COLOQUIAIS
)
stemmer = RSLPStemmer()

def normalizar(texto):
    texto = texto.lower()
    texto = re.sub(r'[^a-zà-ÿ\s]', ' ', texto)   # remove pontuação, números, símbolos
    texto = re.sub(r'\s+', ' ', texto).strip()   # colapsa espaços/quebras de linha
    return texto

## 3. Tokenização e remoção de stopwords

In [4]:
def tokenizar_sem_stopwords(texto_normalizado):
    tokens = word_tokenize(texto_normalizado, language='portuguese')
    return [t for t in tokens if t not in STOPWORDS_PT and len(t) > 1]

## 4. Stemming (RSLP) e lematização (spaCy)

In [5]:
def aplicar_stemming(tokens):
    return [stemmer.stem(t) for t in tokens]

def aplicar_lematizacao(texto_normalizado):
    doc = nlp_pt(texto_normalizado)
    return [tok.lemma_ for tok in doc if tok.text not in STOPWORDS_PT and len(tok.text) > 1]

## 5. Aplicar pipeline em todas as músicas

In [6]:
for musica in musicas:
    normalizado = normalizar(musica['letra'])
    tokens = tokenizar_sem_stopwords(normalizado)

    musica['preprocessamento'] = {
        'texto_normalizado': normalizado,
        'tokens': tokens,
        'tokens_stemizados': aplicar_stemming(tokens),
        'tokens_lematizados': aplicar_lematizacao(normalizado)
    }

print("Pipeline aplicado a", len(musicas), "músicas.")

Pipeline aplicado a 467 músicas.


## 6. Conferência rápida (antes/depois)

In [7]:
exemplo = musicas[0]
print("Título:", exemplo['titulo'], "-", exemplo['artista'])
print("\nTrecho original:\n", exemplo['letra'][:200])
print("\nTokens (sem stopwords):\n", exemplo['preprocessamento']['tokens'][:15])
print("\nStemming:\n", exemplo['preprocessamento']['tokens_stemizados'][:15])
print("\nLematização:\n", exemplo['preprocessamento']['tokens_lematizados'][:15])

Título: Peão Todo Tatuado (part. Jeninho) - Mariana Fagundes

Trecho original:
 Ela não vai pro rodeio pra ver montaria
Ela não vai pro rodeio pra beber cachaça
Sabe o que que ela tá interessada?
Sabe o que que ela quer? Ha-ha-ha

Peão todo tatuado
Beijo de paieiro, perfume de ma

Tokens (sem stopwords):
 ['vai', 'rodeio', 'ver', 'montaria', 'vai', 'rodeio', 'beber', 'cachaça', 'sabe', 'interessada', 'sabe', 'quer', 'ha', 'ha', 'ha']

Stemming:
 ['vai', 'rodei', 'ver', 'mont', 'vai', 'rodei', 'beb', 'cach', 'sab', 'interess', 'sab', 'qu', 'ha', 'ha', 'ha']

Lematização:
 ['ir', 'rodeio', 'ver', 'montario', 'ir', 'rodeio', 'beber', 'cachaça', 'saber', 'interessado', 'saber', 'querer', 'ha', 'ha', 'ha']


## 7. Salvar dataset pré-processado

In [8]:
with open('dataset_musicas_preprocessado.json', 'w', encoding='utf-8') as f:
    json.dump(musicas, f, ensure_ascii=False, indent=2)

print("Salvo em dataset_musicas_preprocessado.json")

Salvo em dataset_musicas_preprocessado.json
